# Figure 6. Trans genetic regulation of glycoproteins and functional characterization

In [ ]:
library(data.table)
library(tidyverse)

## 6a

#### top and right

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork) 
f6a <- image_trim(image_read_pdf("data/Figure6a.pdf", density = 300))  # trim whitespace to reduce top/bottom margin

f6a <- ggdraw() + 
  draw_image(f6a) +
  draw_label(label="", x = 0.5, y = 1, hjust = 0.5, vjust = 1.2, fontface = "bold", size = 16) +
  theme(plot.margin = margin(25, 0, 25, 0))


### 6b (FUCA2 + APOD trans circos)

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)
# Panel b: one FUCA2 trans-glycoQTL locus -> target genes
fuca2_circ <- image_trim(image_read_pdf("data/trans_case_circle.FUCA2.pdf", density = 300))
apod_circ  <- image_trim(image_read_pdf("data/trans_case_circle.APOD.highlight.pdf", density = 300))

bc_title_size <- 13.5
f6b <- ggdraw() + draw_image(fuca2_circ, x = -0.02, scale = 0.90, clip = 'off', valign = 0) + draw_label('FUCA2 trans-glycoQTL hotspot', x = 0.47, y = 0.99, hjust = 0.5, vjust = 1, fontface = 'bold', size = bc_title_size) + theme(plot.margin = margin(0, 0, 0, 0))
f6c <- ggdraw() + draw_image(apod_circ, scale = 0.90, valign = 0) + draw_label('trans-glycoQTLs for APOD', x = 0.50, y = 0.99, hjust = 0.5, vjust = 1, fontface = 'bold', size = bc_title_size) + theme(plot.margin = margin(20, 0, 0, 0))

### 6e

In [ ]:

Figure_6e <- readRDS("data/Figure_6e.rds")
library(dplyr)
line_break <- intToUtf8(10)
trait_levels <- c(
  paste("GRN", "eQTL", sep = line_break),
  paste("APOD", "HexNAc(2)Hex(3)Fuc(1)", sep = line_break),
  paste("APOD", "HexNAc(2)Hex(2)Fuc(1)", sep = line_break)
)
plot_df <- Figure_6e$plot_df %>%
  dplyr::filter(as.character(peptide_ID) %in% trait_levels) %>%
  dplyr::mutate(peptide_ID = factor(as.character(peptide_ID), levels = trait_levels))

In [ ]:

med_df <- plot_df %>%
  group_by(genotype, peptide_ID) %>%
  summarise(median_y = median(pheno_value, na.rm = TRUE), .groups = "drop")

make_panel_label_grob <- function(line1, line2 = NULL, counts = NULL) {
  children <- list(grid::textGrob(
    line1,
    x = grid::unit(0.95, "npc"), y = grid::unit(0.94, "npc"),
    just = c("right", "top"),
    gp = grid::gpar(fontsize = 11.2, fontface = "bold")
  ))
  if (!is.null(line2)) {
    children <- c(children, list(grid::textGrob(
      line2,
      x = grid::unit(0.95, "npc"), y = grid::unit(0.86, "npc"),
      just = c("right", "top"), gp = grid::gpar(fontsize = 11)
    )))
  }
  if (!is.null(counts)) {
    pch <- c(22, 21, 24, 23)
    fill <- c("#2F7ED8", "#68C83E", "#D84A4A", "#9667CF")
    border <- c("#1F4E79", "#2E6B25", "#7F1D1D", "#5B3A8A")
    symbol_x <- 0.92 - rev(seq_along(counts) - 1) * 0.19
    for (ii in seq_along(counts)) {
      children <- c(children, list(
        grid::textGrob(
          paste0(counts[ii], "×"),
          x = grid::unit(symbol_x[ii] - 0.035, "npc"),
          y = grid::unit(0.84, "npc"),
          just = c("right", "centre"), gp = grid::gpar(fontsize = 10.5)
        ),
        grid::pointsGrob(
          x = grid::unit(symbol_x[ii], "npc"), y = grid::unit(0.84, "npc"),
          pch = pch[ii], size = grid::unit(2.7, "mm"),
          gp = grid::gpar(fill = fill[ii], col = border[ii], lwd = 0.8)
        )
      ))
    }
  }
  do.call(grid::grobTree, children)
}

make_genotype_panel <- function(trait, label_grob) {
  panel_df <- plot_df %>% filter(peptide_ID == trait)
  panel_med <- med_df %>% filter(peptide_ID == trait)
  panel_range <- range(panel_df$pheno_value, na.rm = TRUE)
  panel_span <- diff(panel_range)
  if (!is.finite(panel_span) || panel_span <= 0) panel_span <- 1
  panel_limits <- c(
    panel_range[1] - 0.05 * panel_span,
    panel_range[2] + 0.30 * panel_span
  )
  panel_breaks <- scales::breaks_pretty(n = 4)(panel_range)
  panel_breaks <- panel_breaks[
    panel_breaks >= panel_limits[1] & panel_breaks <= panel_range[2]
  ]

  ggplot(panel_df, aes(x = genotype, y = pheno_value)) +
    geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, color = "black") +
    geom_jitter(width = 0.2, size = 2, alpha = 0.6, aes(color = genotype)) +
    geom_line(
      data = panel_med,
      aes(x = as.numeric(genotype), y = median_y, group = 1),
      inherit.aes = FALSE, color = "grey50", linewidth = 1, linetype = "dashed"
    ) +
    scale_color_manual(values = Figure_6e$geno_colors) +
    scale_y_continuous(limits = panel_limits, breaks = panel_breaks, expand = c(0, 0)) +
    annotation_custom(label_grob, xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf) +
    labs(
      x = "Genotype (rs5848)",
      y = "Normalized abundance"
    ) +
    theme_bw(base_size = 14) +
    theme(
      axis.text.x = element_text(size = 14, face = "bold"),
      axis.text.y = element_text(size = 14),
      axis.title = element_text(size = 16, face = "bold"),
      axis.title.y = element_text(size = 15, face = "bold", margin = margin(r = 5)),
      legend.position = "none",
      panel.grid = element_blank(),
      panel.border = element_rect(linewidth = 1.2),
      plot.margin = margin(t = 3, r = 3, b = 3, l = 3)
    )
}

f6e_grn <- make_genotype_panel(
  trait_levels[1], make_panel_label_grob("GRN", "RNA")
)
f6e_glycan_1 <- make_genotype_panel(
  trait_levels[2], make_panel_label_grob("APOD N98", counts = c(2, 3, 1))
)
f6e_glycan_2 <- make_genotype_panel(
  trait_levels[3], make_panel_label_grob("APOD N98", counts = c(2, 2, 1))
)
f6e <- patchwork::wrap_plots(f6e_grn, f6e_glycan_1, f6e_glycan_2, nrow = 1) +
  patchwork::plot_layout(axis_titles = "collect")

### 6f

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)
ncam1_draft <- image_trim(image_read_pdf("data/APOD.pdf", density = 300))  # trim whitespace to reduce panel margin

# Draw image-based plots (f6c, f6e)
f6f <- ggdraw() + 
  draw_image(ncam1_draft) +
  theme(plot.margin = margin(30, 22, 22, 22))

### 6g

In [ ]:
conflicted::conflicts_prefer(base::setdiff)
conflicted::conflicts_prefer(base::intersect)
conflicted::conflicts_prefer(base::union)
conflicted::conflicts_prefer(base::setequal)


source("../Figure4/data/colocboost_plot.R")
Figure_6g <- readRDS("data/Figure_6g.rds")
start <- 5341
end <- 5619

glyph <- function(count, pch, fill, border) {
  list(count = count, pch = pch, fill = fill, border = border)
}
glycan_annotation <- function(gene, site, hexnac, hex, fuc = 0, neuac = 0) {
  parts <- list(
    glyph(hexnac, 22, "#2F7ED8", "#1F4E79"),
    glyph(hex, 21, "#68C83E", "#2E6B25")
  )
  if (fuc > 0) parts <- c(parts, list(glyph(fuc, 24, "#D84A4A", "#7F1D1D")))
  if (neuac > 0) parts <- c(parts, list(glyph(neuac, 23, "#9667CF", "#5B3A8A")))
  list(line1 = paste(gene, site), glycan = parts, component_gap = 0.15, symbol_offset = 0.105, glycan_y_offset = 0.145)
}

outcome_annotations <- vector("list", length(Figure_6g$new_id_vector))
outcome_annotations[[42]] <- list(line1 = "ADRD", line2 = "GWAS")
outcome_annotations[[4]] <- list(line1 = "GRN", line2 = "RNA")
outcome_annotations[[5]] <- list(line1 = "GRN", line2 = "Exc RNA")
outcome_annotations[[9]] <- list(line1 = "GRN", line2 = "Oli RNA")
outcome_annotations[[41]] <- list(line1 = "APOD", line2 = "Protein")
outcome_annotations[[11]] <- glycan_annotation("APOD", "N98", 2, 3, 1)
outcome_annotations[[35]] <- glycan_annotation("APOD", "N98", 2, 2, 1)
highlight_pos <- 44352876

draw_gene_track <- function(x_range, tick_at, column, axis_size) {
  plot.new()
  plot.window(xlim = c(44300000, 44400000), ylim = c(-0.35, 0.35), xaxs = "i", yaxs = "i")
  with(Figure_6g$intron_df, arrows(
    start, 0, end, 0, length = 0.045, angle = 25, code = 2, lwd = 0.7
  ))
  with(Figure_6g$ex_df, rect(
    start, -0.18, end, 0.18, col = "steelblue", border = NA
  ))
  display_ticks <- c(tick_at[1], tick_at[length(tick_at)])
  display_labels <- paste0(formatC(display_ticks / 1e6, format = "f", digits = 2), "M")
  axis(1, at = display_ticks, labels = FALSE)
  coordinate_cex <- axis_size * 0.78
  mtext(display_labels[1], side = 1, line = 0.85, at = display_ticks[1], adj = 0, cex = coordinate_cex)
  mtext(display_labels[2], side = 1, line = 0.85, at = display_ticks[2], adj = 1, cex = coordinate_cex)
  mtext("GRN", side = 1, line = 2.55, cex = 1.45, font = 2)
  box(lwd = 0.8)
}

In [ ]:

tmp <- tempfile(fileext = ".png")
draw_figure_6f <- function() {
  colocboost_plot(
    Figure_6g$cb_res,
    outcome_names = Figure_6g$new_id_vector,
    grange = start:end,
    xlim = c(44300000, 44400000),
    xaxs = "i",
    outcome_idx = c(42, 4, 5, 9, 41, 11, 35),
    variant_coord = TRUE,
    xlab = "",
    outcome_annotations = outcome_annotations,
    bottom_track = draw_gene_track,
    compact_x_axes = TRUE,
    track_height = 0.40,
    panel_y_ticks = list('5' = c(0, 4, 8, 12)),
    panel_left_margin = 5.3,
    panel_right_margin = 2.0,
    panel_inner_right_margin = 0.3,
    add_highlight = TRUE,
    add_highlight_idx = highlight_pos,
    add_highlight_style = "coral_circle",
    outcome_legend_size = 1.5,
    annotation_scale = 1.45,
    lab_style = c(1.90, 1),
    axis_style = c(1.65, 1),
    genomic_ticks = c(44300000, 44400000),
    show_cos_to_uncoloc = FALSE,
    plot_focal_only = TRUE
  )
}
png(tmp, width = 2480, height = 2930, res = 300)
draw_figure_6f()
dev.off()

library(magick)
f6g_img <- image_read(tmp)
f6g_a <- ggdraw() +
  draw_image(f6g_img) +
  theme(plot.margin = margin(0, 0, 0, 0))

f6g_a

# Gene tracks are rendered inside f6g_a with shared genomic coordinates
invisible(NULL)

# Gene tracks are rendered inside f6g_a with shared genomic coordinates
invisible(NULL)

## Combine

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)

# Column assemblies are defined in the final layout cell below
invisible(NULL)

# Final Figure 6 layout: a | b-c-d
panel_tag_theme <- theme(
  plot.tag = element_text(size = 22, face = "bold"),
  plot.tag.position = c(0, 1)
)

e_panel_tag_theme <- theme(
  plot.tag = element_text(size = 22, face = "bold"),
  plot.tag.position = c(-0.08, 1)
)

bcd_col <- wrap_plots(
  wrap_elements(f6b) + labs(tag = "b") + panel_tag_theme,
  wrap_elements(f6c) + labs(tag = "c") + panel_tag_theme,
  wrap_elements(f6f) + labs(tag = "d") + panel_tag_theme,
  ncol = 1,
  heights = c(0.83, 0.69, 0.62)
)

ef_col <- wrap_plots(
  wrap_elements(f6e) + labs(tag = "e") + e_panel_tag_theme,
  wrap_elements(f6g_a) + labs(tag = "f") + panel_tag_theme,
  ncol = 1,
  heights = c(0.52, 1.55)
)

pdf('Figure6.pdf', width = 21.6, height = 11.5)
print(wrap_plots(
  wrap_elements(f6a) + labs(tag = "a") + panel_tag_theme,
  bcd_col,
  ef_col,
  nrow = 1,
  widths = c(1.45, 0.72, 1.11)
))
dev.off()
combined_fig <- image_read_pdf('Figure6.pdf', density = 100)  # high resolution

options(repr.plot.width = 20, repr.plot.height = 16)
ggdraw() + 
  draw_image(combined_fig) +
  theme(plot.margin = margin(0, 0, 0, 0))